<a href="https://colab.research.google.com/github/danyComit/Arduino-Raqi-Noise-Analysis-Tool/blob/main/TransformerDistillBert.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:


# 1) Install dependencies
!pip -q install -U transformers datasets accelerate scikit-learn sentencepiece

# 2) Imports
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

# 3) Load dataset
# Upload your file in Colab left panel (Files), then keep this filename exactly.
DATA_PATH = "/content/propaganda_data copy.csv"

df = pd.read_csv(DATA_PATH)
df.columns = df.columns.str.strip()
df = df[["text", "label"]].dropna().copy()
df["text"] = df["text"].astype(str).str.strip()
df = df[df["text"].str.len() > 0]
df["label"] = df["label"].astype(int)

print("Dataset shape:", df.shape)
print("Label counts:\n", df["label"].value_counts())

# 4) Split dataset (stratified)
train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df["label"],
    random_state=42
)

print("\nTrain label counts:\n", train_df["label"].value_counts())
print("\nVal label counts:\n", val_df["label"].value_counts())
print("\nTest label counts:\n", test_df["label"].value_counts())

# 5) Balance only train set by oversampling minority
label_counts = train_df["label"].value_counts()
majority_label = int(label_counts.idxmax())
minority_label = int(label_counts.idxmin())

majority = train_df[train_df["label"] == majority_label]
minority = train_df[train_df["label"] == minority_label]

minority_oversampled = minority.sample(
    n=len(majority),
    replace=True,
    random_state=42
)

train_df_balanced = pd.concat([majority, minority_oversampled]).sample(
    frac=1, random_state=42
).reset_index(drop=True)

print("\nBalanced train counts:\n", train_df_balanced["label"].value_counts())
print("Majority label:", majority_label, "| Minority label:", minority_label)

# 6) HF datasets + tokenization
model_name = "distilbert-base-uncased"
max_length = 256

tokenizer = AutoTokenizer.from_pretrained(model_name)

train_ds = Dataset.from_pandas(train_df_balanced.reset_index(drop=True))
val_ds = Dataset.from_pandas(val_df.reset_index(drop=True))
test_ds = Dataset.from_pandas(test_df.reset_index(drop=True))

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding=False,
        max_length=max_length
    )

train_ds = train_ds.map(tokenize_function, batched=True)
val_ds = val_ds.map(tokenize_function, batched=True)
test_ds = test_ds.map(tokenize_function, batched=True)

keep_cols = ["input_ids", "attention_mask", "label"]
train_ds = train_ds.remove_columns([c for c in train_ds.column_names if c not in keep_cols])
val_ds = val_ds.remove_columns([c for c in val_ds.column_names if c not in keep_cols])
test_ds = test_ds.remove_columns([c for c in test_ds.column_names if c not in keep_cols])

train_ds.set_format("torch")
val_ds.set_format("torch")
test_ds.set_format("torch")

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

# 7) Metrics (explicit per class to avoid confusing recall=0 issues)
# If propaganda is class 1, keep this as 1. If propaganda is class 0, change to 0.
PROPAGANDA_LABEL = 1

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    precision_per_class, recall_per_class, f1_per_class, _ = precision_recall_fscore_support(
        labels, preds, labels=[0, 1], zero_division=0
    )

    idx = 0 if PROPAGANDA_LABEL == 0 else 1

    return {
        "accuracy": accuracy_score(labels, preds),
        "precision_propaganda": float(precision_per_class[idx]),
        "recall_propaganda": float(recall_per_class[idx]),
        "f1_propaganda": float(f1_per_class[idx]),
        "recall_class_0": float(recall_per_class[0]),
        "recall_class_1": float(recall_per_class[1]),
        "f1_macro": float(np.mean(f1_per_class)),
    }

# 8) Training config
training_args = TrainingArguments(
    output_dir="/content/transformer_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=5e-6,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=6,
    weight_decay=0.01,
    max_grad_norm=1.0,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

# 9) Train
trainer.train()





Torch: 2.10.0+cu128
CUDA available: True
Dataset shape: (19570, 2)
Label counts:
 label
0    14523
1     5047
Name: count, dtype: int64

Train label counts:
 label
0    11618
1     4038
Name: count, dtype: int64

Val label counts:
 label
0    1452
1     505
Name: count, dtype: int64

Test label counts:
 label
0    1453
1     504
Name: count, dtype: int64

Balanced train counts:
 label
0    11618
1    11618
Name: count, dtype: int64
Majority label: 0 | Minority label: 1


Map:   0%|          | 0/23236 [00:00<?, ? examples/s]

Map:   0%|          | 0/1957 [00:00<?, ? examples/s]

Map:   0%|          | 0/1957 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=17430, training_loss=0.2924927388123924, metrics={'train_runtime': 916.829, 'train_samples_per_second': 152.063, 'train_steps_per_second': 19.011, 'total_flos': 1957594938125952.0, 'train_loss': 0.2924927388123924, 'epoch': 6.0})

In [8]:
print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best metric:", trainer.state.best_metric)
print("Current model source:", trainer.model.name_or_path)

# 10) Final test evaluation
pred_out = trainer.predict(test_ds)
test_preds = np.argmax(pred_out.predictions, axis=1)
test_labels = pred_out.label_ids

print("\nConfusion matrix (rows=true, cols=pred):")
print(confusion_matrix(test_labels, test_preds, labels=[0, 1]))

print("\nClassification report:")
print(classification_report(test_labels, test_preds, labels=[0, 1], zero_division=0))

print("\nPrediction distribution:")
print(pd.Series(test_preds).value_counts().sort_index())

# Optional: metrics dict directly from predictions
test_metrics = compute_metrics((pred_out.predictions, test_labels))
print("\nComputed test metrics:", test_metrics)

Best checkpoint: /content/transformer_results/checkpoint-5810
Best metric: 0.7271790847108451
Current model source: distilbert-base-uncased



Confusion matrix (rows=true, cols=pred):
[[1219  234]
 [ 163  341]]

Classification report:
              precision    recall  f1-score   support

           0       0.88      0.84      0.86      1453
           1       0.59      0.68      0.63       504

    accuracy                           0.80      1957
   macro avg       0.74      0.76      0.75      1957
weighted avg       0.81      0.80      0.80      1957


Prediction distribution:
0    1382
1     575
Name: count, dtype: int64

Computed test metrics: {'accuracy': 0.797138477261114, 'precision_propaganda': 0.5930434782608696, 'recall_propaganda': 0.6765873015873016, 'f1_propaganda': 0.6320667284522706, 'recall_class_0': 0.8389538885065382, 'recall_class_1': 0.6765873015873016, 'f1_macro': 0.746015727541832}


In [6]:
print(trainer.state.best_model_checkpoint)


/content/transformer_results/checkpoint-5810


In [7]:
import json, glob, os

ckpts = sorted(glob.glob("/content/transformer_results/checkpoint-*"), key=lambda p: int(p.split("-")[-1]))
for c in ckpts:
    s = json.load(open(os.path.join(c, "trainer_state.json")))
    print(os.path.basename(c), "epoch =", s.get("epoch"), "| global_step =", s.get("global_step"))


checkpoint-2905 epoch = 1.0 | global_step = 2905
checkpoint-5810 epoch = 2.0 | global_step = 5810
checkpoint-8715 epoch = 3.0 | global_step = 8715
checkpoint-11620 epoch = 4.0 | global_step = 11620
checkpoint-14525 epoch = 5.0 | global_step = 14525
checkpoint-17430 epoch = 6.0 | global_step = 17430
